# 🎵 GTZAN Music Genre Classification — EDA

Exploratory Data Analysis on the GTZAN dataset:
- Class distribution
- Feature statistics
- Audio waveform & spectrogram visualization
- Correlation analysis

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import librosa
import librosa.display
import yaml
import warnings
warnings.filterwarnings('ignore')

# Config
ROOT = os.path.abspath('..')
with open(os.path.join(ROOT, 'config.yaml'), 'r') as f:
    CFG = yaml.safe_load(f)

DATASET_PATH = os.path.join(ROOT, CFG['data']['dataset_path'])
CSV_PATH = os.path.join(ROOT, CFG['data']['csv_path'])
GENRES = CFG['data']['genres']
SR = CFG['audio']['sample_rate']

print(f'Genres: {GENRES}')
print(f'Dataset path: {DATASET_PATH}')

## 1. Class Distribution

In [ ]:
# Count files per genre
genre_counts = {}
for genre in GENRES:
    gdir = os.path.join(DATASET_PATH, genre)
    if os.path.isdir(gdir):
        genre_counts[genre] = len([f for f in os.listdir(gdir) if f.endswith('.wav')])
    else:
        genre_counts[genre] = 0

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(genre_counts.keys(), genre_counts.values(), color=sns.color_palette('Set2', 10))
ax.set_xlabel('Genre')
ax.set_ylabel('# Tracks')
ax.set_title('GTZAN — Tracks per Genre')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()
print(genre_counts)

## 2. Tabular Feature Overview

In [ ]:
# Load pre-extracted features (GTZAN features_30_sec.csv or our extracted CSV)
extracted = os.path.join(ROOT, 'outputs', 'features_extracted.csv')
csv = extracted if os.path.exists(extracted) else CSV_PATH

if os.path.exists(csv):
    df = pd.read_csv(csv)
    print(f'Loaded {csv}')
    print(f'Shape: {df.shape}')
    display(df.head())
    display(df.describe())
else:
    print(f'No CSV found at {csv}. Run extract_features.py first.')
    df = None

## 3. Feature Distributions by Genre

In [ ]:
if df is not None:
    # Box plots for selected features
    features_to_plot = ['spectral_centroid_mean', 'spectral_rolloff_mean',
                        'zcr_mean', 'rms_mean', 'mfcc1_mean', 'mfcc2_mean']
    # Fallback if column names differ
    features_to_plot = [f for f in features_to_plot if f in df.columns]
    if not features_to_plot:
        features_to_plot = [c for c in df.select_dtypes(include='number').columns[:6]]

    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    for i, feat in enumerate(features_to_plot[:6]):
        ax = axes[i // 3, i % 3]
        sns.boxplot(data=df, x='label', y=feat, ax=ax, palette='Set2')
        ax.set_xticklabels(ax.get_xticklabels(), rotation=45)
        ax.set_title(feat)
    plt.suptitle('Feature Distributions by Genre', fontsize=14)
    plt.tight_layout()
    plt.show()

## 4. Correlation Heatmap

In [ ]:
if df is not None:
    numeric = df.select_dtypes(include='number')
    corr = numeric.corr()
    fig, ax = plt.subplots(figsize=(16, 14))
    sns.heatmap(corr, cmap='coolwarm', center=0, ax=ax,
                xticklabels=False, yticklabels=False)
    ax.set_title('Feature Correlation Matrix')
    plt.tight_layout()
    plt.show()

## 5. Waveform & Mel-Spectrogram Samples

In [ ]:
# Show one waveform + spectrogram per genre
fig, axes = plt.subplots(len(GENRES), 2, figsize=(16, 3 * len(GENRES)))

for i, genre in enumerate(GENRES):
    gdir = os.path.join(DATASET_PATH, genre)
    if not os.path.isdir(gdir):
        continue
    wavs = sorted([f for f in os.listdir(gdir) if f.endswith('.wav')])
    if not wavs:
        continue
    fpath = os.path.join(gdir, wavs[0])
    y, sr = librosa.load(fpath, sr=SR, duration=5)

    # Waveform
    ax_wave = axes[i, 0]
    librosa.display.waveshow(y, sr=sr, ax=ax_wave, color='steelblue')
    ax_wave.set_title(f'{genre} — Waveform')
    ax_wave.set_ylabel('')

    # Mel-Spectrogram
    ax_spec = axes[i, 1]
    S = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=128)
    S_dB = librosa.power_to_db(S, ref=np.max)
    librosa.display.specshow(S_dB, sr=sr, x_axis='time', y_axis='mel',
                             ax=ax_spec, cmap='viridis')
    ax_spec.set_title(f'{genre} — Mel-Spectrogram')

plt.tight_layout()
plt.show()

## 6. MFCC Visualization

In [ ]:
# Show MFCCs for one track per genre (first 3 genres for conciseness)
fig, axes = plt.subplots(3, 1, figsize=(14, 9))
for i, genre in enumerate(GENRES[:3]):
    gdir = os.path.join(DATASET_PATH, genre)
    if not os.path.isdir(gdir):
        continue
    wavs = sorted([f for f in os.listdir(gdir) if f.endswith('.wav')])
    if not wavs:
        continue
    y, sr = librosa.load(os.path.join(gdir, wavs[0]), sr=SR, duration=5)
    mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=40)
    librosa.display.specshow(mfccs, sr=sr, x_axis='time', ax=axes[i])
    axes[i].set_title(f'{genre} — MFCCs')
    axes[i].set_ylabel('MFCC')

plt.tight_layout()
plt.show()

## 7. t-SNE / PCA of Tabular Features

In [ ]:
if df is not None:
    from sklearn.preprocessing import StandardScaler, LabelEncoder
    from sklearn.manifold import TSNE

    numeric = df.select_dtypes(include='number')
    X = StandardScaler().fit_transform(numeric)
    labels = df['label']

    tsne = TSNE(n_components=2, random_state=42, perplexity=30)
    X_2d = tsne.fit_transform(X)

    fig, ax = plt.subplots(figsize=(10, 8))
    for genre in GENRES:
        mask = labels == genre
        ax.scatter(X_2d[mask, 0], X_2d[mask, 1], label=genre, alpha=0.7, s=30)
    ax.legend()
    ax.set_title('t-SNE of Extracted Audio Features')
    plt.tight_layout()
    plt.show()

---
**Next steps:** Run the training scripts to build and compare models.